In [14]:
import numpy as np
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from tqdm import tqdm
import math

base = "/Volumes/Extreme SSD/stream_stead/data_stead/"

X = np.load(base + "X_multi_ac3.npy")      # (22527, 6000, 3)
y = np.load(base + "y_multi_ac3.npy")      # (22527,)
y = y.reshape(-1, 1)

X = mx.array(X)
y = mx.array(y)

split = int(0.8 * X.shape[0])
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)


Train: (18021, 6000, 3) (18021, 1)
Val: (4506, 6000, 3) (4506, 1)


In [15]:
class Conv1dFFT(nn.Module):
    def __init__(self, in_ch, out_ch, k):
        super().__init__()
        self.weight = mx.random.normal((out_ch, in_ch, k)) * 0.01
        self.bias = mx.zeros((out_ch,))
        self.k = k

    def __call__(self, x):
        # x: (B, L, C)
        B, L, C = x.shape
        K = self.k

        fft_size = L + K - 1

        # FFT input
        Xf = mx.fft.rfft(x, n=fft_size, axis=1)  # (B, F, C)

        # Pad kernel
        w = mx.pad(self.weight, ((0,0),(0,0),(0, L-K)))
        Wf = mx.fft.rfft(w, n=fft_size, axis=2)  # (out, in, F)

        # Multiply in frequency domain
        Yf = mx.zeros((B, Xf.shape[1], self.weight.shape[0]))
        for oc in range(self.weight.shape[0]):
            for ic in range(C):
                Yf[:, :, oc] += Xf[:, :, ic] * Wf[oc, ic, :]

        # IFFT
        y = mx.fft.irfft(Yf, n=fft_size, axis=1)

        # Crop to original length
        y = y[:, :L, :]

        # Add bias
        y = y + self.bias.reshape(1, 1, -1)

        return y


In [16]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k):
        super().__init__()
        self.conv = Conv1dFFT(in_ch, out_ch, k)
        self.act = nn.ReLU()

    def __call__(self, x):
        return self.act(self.conv(x))

class AC3_MLX(nn.Module):
    def __init__(self):
        super().__init__()

        self.b1 = ConvBlock(3, 32, 7)
        self.p1 = nn.MaxPool1d(4)

        self.b2 = ConvBlock(32, 64, 5)
        self.p2 = nn.MaxPool1d(4)

        self.b3 = ConvBlock(64, 128, 5)
        self.p3 = nn.MaxPool1d(4)

        self.b4 = ConvBlock(128, 256, 3)

        self.fc1 = nn.Linear(256, 128)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 1)
        self.sig = nn.Sigmoid()

    def __call__(self, x):
        x = self.b1(x); x = self.p1(x)
        x = self.b2(x); x = self.p2(x)
        x = self.b3(x); x = self.p3(x)
        x = self.b4(x)

        x = mx.mean(x, axis=1)  # GAP

        x = self.fc1(x)
        x = self.act1(x)
        x = self.fc2(x)
        return self.sig(x)

model = AC3_MLX()


In [17]:
opt = optim.Adam(1e-4)

def bce_loss(pred, y):
    eps = 1e-7
    pred = mx.clip(pred, eps, 1 - eps)
    return -mx.mean(y * mx.log(pred) + (1 - y) * mx.log(1 - pred))

def loss_fn(params, model, x, y):
    model.update(params)
    pred = model(x)
    return bce_loss(pred, y)

params = model.trainable_parameters()
loss_and_grad = mx.value_and_grad(loss_fn)


In [18]:
batch_size = 32
epochs = 5

N = X_train.shape[0]
steps = math.ceil(N / batch_size)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    idx = mx.random.permutation(mx.arange(N))
    pbar = tqdm(range(steps), desc="Training")

    for step in pbar:
        bidx = idx[step*batch_size:(step+1)*batch_size]
        xb = X_train[bidx]
        yb = y_train[bidx]

        loss, grads = loss_and_grad(params, model, xb, yb)
        params = opt.update(params, grads)

        pbar.set_postfix({"loss": float(loss.item())})

    model.update(params)
    val_pred = model(X_val)
    val_loss = bce_loss(val_pred, y_val)
    print(f"Validation loss: {float(val_loss.item()):.4f}")



Epoch 1/5


Training:   0%|          | 1/564 [00:05<48:07,  5.13s/it, loss=0.693]


ValueError: [tree_flatten] The argument should contain only arrays